<a href="https://colab.research.google.com/github/gitmystuff/DTSC4050/blob/main/10-Regression_I/Your_Name_Regression_I_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Dummy Variables, OLS Coefficients, and the Baseline Group
Your Name

**Time:** ~60 minutes
**Goal:** Understand how a categorical predictor gets encoded as dummy variables in a
regression, and — most importantly — how to read the `statsmodels` OLS summary table:
what the intercept means, what each coefficient means *relative to a baseline group*,
and how that compares to a simple continuous-predictor regression fit with `scikit-learn`.

**How this works:** Most cells are filled in for you to run and study. Cells marked
`# TODO` need you to fill in a line or two. Markdown cells with **Q:** are short-answer
questions — answer them in the cell right below in a sentence or two.


## Getting Started

* Colab - get notebook from gitmystuff DTSC4050 repository
* Save a Copy in Drive
* Remove Copy of
* Edit name
* Clean up Colab Notebooks folder
* Submit shared link


## Part 1: Setup (5 min)

We'll simulate a dataset instead of using a fixed example, so the numbers won't match
anything you've already seen in lecture — you'll need to actually read the output rather
than recall it.

**Scenario:** A company tests three employee training programs (A, B, C) and measures a
productivity score afterward. We want to know: does the training program a person
received affect their productivity score?


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

np.random.seed(42)  # keep this fixed so everyone gets the same numbers

n_per_group = 12

program_A = np.random.normal(loc=70, scale=5, size=n_per_group)
program_B = np.random.normal(loc=76, scale=5, size=n_per_group)
program_C = np.random.normal(loc=65, scale=5, size=n_per_group)

data = pd.DataFrame({
    'score': np.concatenate([program_A, program_B, program_C]),
    'program': ['A'] * n_per_group + ['B'] * n_per_group + ['C'] * n_per_group
})

data.head()


In [ ]:
# TODO: use .groupby() to find the mean score for each program
# This is your "ground truth" -- keep these numbers in mind for Part 2.
group_means = data.groupby('program')['score'].mean()
print(group_means)


## Part 2: Dummy Coding and `smf.ols` (20 min)

`program` is categorical (A, B, C) — a regression needs numbers, not letters. We convert
it into **dummy variables**: new 0/1 columns, one per category, with one category left
out as the **baseline** (a.k.a. reference group).


In [ ]:
# Dummy-code the categorical predictor, dropping the first category
# (dtype=int keeps the new columns as 0/1 integers, not True/False booleans,
# which keeps the column names clean in the regression formula below)
reg_data = pd.get_dummies(data, columns=['program'], drop_first=True, dtype=int)
print(reg_data.head())


**Q1:** Which category got left out (the baseline), and which two columns were
created? *(Hint: look at the column names above.)*


_Your answer:_

In [ ]:
# Fit the regression: score predicted from the two dummy variables
ols_model = smf.ols('score ~ program_B + program_C', data=reg_data).fit()
print(ols_model.summary())


### Reading the table

Focus on the `coef` column in the middle of the summary:

| Term | What it represents |
|---|---|
| `Intercept` | The **mean score of the baseline group** (the category we dropped) |
| `program_B` | How much B's mean score **differs from the baseline** |
| `program_C` | How much C's mean score **differs from the baseline** |

So the predicted mean for the baseline group is just the intercept. The predicted mean
for group B is `Intercept + program_B`. The predicted mean for group C is
`Intercept + program_C`.


In [ ]:
# TODO: use the model's coefficients to reconstruct each group's predicted mean.
# ols_model.params is a Series -- print it first if you want to see the labels.
intercept = ols_model.params['Intercept']
coef_B = ols_model.params['program_B']
coef_C = ols_model.params['program_C']

predicted_baseline = intercept
predicted_B = intercept + coef_B
predicted_C = intercept + coef_C

print(f"Predicted baseline group mean: {predicted_baseline:.2f}")
print(f"Predicted B group mean: {predicted_B:.2f}")
print(f"Predicted C group mean: {predicted_C:.2f}")


**Q2:** Compare these three predicted means to the `group_means` you calculated in
Part 1 with `.groupby()`. Do they match? Should they? Why or why not?


_Your answer:_

**Q3:** Look at the `P>|t|` column for `program_B` and `program_C`. Which coefficients
are statistically significant at $\alpha = 0.05$? What does a significant coefficient tell
you about that group compared to the baseline specifically (not compared to the other
non-baseline group)?


_Your answer:_

### Changing the baseline

The baseline is chosen automatically (alphabetically, by default) — but it's arbitrary.
We can force a different group to be the reference using `C(..., Treatment(reference=...))`
in the formula.


In [ ]:
# TODO: refit the model using program C as the baseline instead of A
ols_model_C_baseline = smf.ols(
    "score ~ C(program, Treatment(reference='C'))",
    data=data
).fit()
print(ols_model_C_baseline.summary())


**Q4:** Did the R-squared or F-statistic change when you switched the baseline group?
Did the *individual* coefficients and their meaning change? What does that tell you about
what the baseline choice does and doesn't affect?


_Your answer:_

## Part 3: A Continuous Predictor with `scikit-learn` (15 min)

Now let's step away from categories entirely. Suppose instead we track how many hours of
training each employee received (a continuous variable) rather than which program they
were in.


In [ ]:
np.random.seed(7)

hours = np.random.uniform(1, 10, size=30)
noise = np.random.normal(0, 4, size=30)
score = 50 + 3.2 * hours + noise  # true relationship: score = 50 + 3.2*hours + error

sk_data = pd.DataFrame({'hours': hours, 'score': score})
sk_data.head()


In [ ]:
from sklearn.linear_model import LinearRegression

x_reshaped = sk_data['hours'].values.reshape(-1, 1)
y = sk_data['score'].values

sk_model = LinearRegression()
sk_model.fit(x_reshaped, y)
print(f'y = {sk_model.intercept_:.2f} + {sk_model.coef_[0]:.2f}(X)')


In [ ]:
# TODO: plot the data and the fitted line
plt.scatter(sk_data['hours'], sk_data['score'])
plt.plot(sk_data['hours'], sk_model.predict(x_reshaped), color='red')
plt.xlabel('Hours of Training')
plt.ylabel('Productivity Score')
plt.title('Hours of Training vs. Productivity Score')
plt.show()


**Q5:** In Part 2, the intercept represented the mean of a *baseline group*. Here,
with a continuous predictor, what does the intercept represent instead? *(Hint: what
value would `hours` need to be for the model to just predict the intercept?)*


_Your answer:_

**Q6:** The `sk_model.coef_[0]` value is the slope. In plain language, what does it
mean in terms of productivity score per additional hour of training?


_Your answer:_

## Part 4: Wrap-Up (10-15 min)

**Q7:** `smf.ols().summary()` and `sklearn`'s `LinearRegression` both fit a least-squares
regression line — but the outputs look very different. Name **two** pieces of information
that `statsmodels`' summary table gives you that the basic `sklearn` fit (intercept +
coefficients) does not.


_Your answer:_

**Q8:** Suppose you fit the categorical model from Part 2 (`score ~ program_B +
program_C`) and, separately, calculated group means directly with `.groupby()`. You
should have found in Q2 that these matched exactly. Why does a regression on dummy
variables produce *exactly* the same numbers as group means — what is the regression
actually doing under the hood when every predictor is 0/1?


_Your answer:_

**Q9 (short reflection):** In one or two sentences, explain what "baseline group"
means to someone who has never seen a regression summary table before.


_Your answer:_

---
### Submission
Make sure all cells run top to bottom without errors (**Runtime → Run all** in Colab),
then save/download and submit per the course instructions.
